# Learn Human-in-the-Loop (HITL) in LangGraph

This notebook teaches you HITL step by step using separate cells.
Run each cell one at a time and read the output before moving to the next.

**3 key concepts:**
- `interrupt(value)` — pauses the graph and sends `value` back to you
- `Command(resume=answer)` — sends your answer back into the paused graph
- `MemorySaver` — saves the paused state so the graph can resume later

**The flow:**
```
You ask a question
  → AI generates a draft answer
    → Graph PAUSES (interrupt)
      → You review the draft
        → You approve or edit
          → Graph RESUMES (Command)
            → Final answer is produced
```

In [1]:
# Cell 1: Imports and setup
from langgraph.graph import StateGraph, END
from langgraph.types import Command, interrupt
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict
import builtins
print('All imports loaded successfully.')

All imports loaded successfully.


## Step 1: Define the State and Graph Nodes

The `State` holds three fields that flow through the graph.
- `generate_draft` calls a fake AI to create a draft answer.
- `human_review` calls `interrupt()` which **pauses the entire graph** and waits for you.

In [2]:
# Cell 2: Define the state and two graph nodes

class State(TypedDict):
    question: str
    draft_answer: str
    final_answer: str

def generate_draft(state: State):
    """Node 1: The AI creates a draft answer."""
    draft = f"AI thinks the answer to '{state['question']}' is: LangGraph helps build stateful AI workflows."
    print(f'[generate_draft] Created draft: {draft}')
    return {'draft_answer': draft}

def human_review(state: State):
    """Node 2: Pause the graph here. The human must approve or edit before continuing."""
    # interrupt() STOPS the graph and sends this message back to whoever called app.invoke()
    # The graph will not continue past this line until you resume it with Command(resume=...)
    human_response = interrupt({
        'message': 'Please review the AI draft below.',
        'draft': state['draft_answer'],
        'instructions': "Type 'approve' to accept, or type a new answer to replace the draft."
    })
    print(f'[human_review] Human responded with: {human_response}')
    if human_response.strip().lower() == 'approve':
        return {'final_answer': state['draft_answer']}
    return {'final_answer': human_response}

print('State and nodes defined.')

def format_answer(state: State):
    return {'final_answer': f'\n{state["final_answer"]}\n'}

State and nodes defined.


## Step 2: Build and Compile the Graph

We connect the nodes: `generate_draft` → `human_review` → `END`.

**Important:** We pass `checkpointer=MemorySaver()` so the graph can save its paused state.
Without a checkpointer, `interrupt()` would not work because there's nowhere to save the pause point.

In [3]:
# Cell 3: Build the graph

graph = StateGraph(State)
graph.add_node('generate_draft', generate_draft)
graph.add_node('human_review', human_review)
graph.add_node('format_answer', format_answer)  # Add a formatting step after human review
graph.set_entry_point('generate_draft')
graph.add_edge('generate_draft', 'human_review')
graph.add_edge('human_review', 'format_answer')
graph.add_edge('format_answer', END)

# Compile with a checkpointer (required for interrupt/resume to work)
app = graph.compile(checkpointer=MemorySaver())
print('Graph compiled. Ready to run.')

Graph compiled. Ready to run.


## Step 3: Run the Graph (it will PAUSE here!)

When you run the next cell, the graph will:
1. Execute `generate_draft` and create a draft answer.
2. Enter `human_review` and hit `interrupt()` which **pauses the graph**.
3. Return the result with `__interrupt__` metadata so you can see the draft.

**Notice:** The graph does NOT finish. It is waiting for you.

In [4]:
# Cell 4: Start the graph - it will PAUSE at interrupt()

# Each HITL session needs a unique thread_id so the checkpointer can track it
config = {'configurable': {'thread_id': 'my-first-hitl'}}

# Run the graph with a question
result = app.invoke({'question': 'What is LangGraph?'}, config=config)

# Check: did the graph pause?
if '__interrupt__' in result:
    print('\n=== GRAPH IS PAUSED ===')
    print('The AI created a draft. Here is what interrupt() sent back to you:\n')
    interrupt_data = result['__interrupt__'][0].value
    print(f"  Message: {interrupt_data['message']}")
    print(f"  Draft: {interrupt_data['draft']}")
    print(f"  Instructions: {interrupt_data['instructions']}")
    print('\n=== Now run the NEXT cell to approve or edit the draft. ===')
else:
    print('Graph finished without pausing (this should not happen in this example).')

[generate_draft] Created draft: AI thinks the answer to 'What is LangGraph?' is: LangGraph helps build stateful AI workflows.

=== GRAPH IS PAUSED ===
The AI created a draft. Here is what interrupt() sent back to you:

  Message: Please review the AI draft below.
  Draft: AI thinks the answer to 'What is LangGraph?' is: LangGraph helps build stateful AI workflows.
  Instructions: Type 'approve' to accept, or type a new answer to replace the draft.

=== Now run the NEXT cell to approve or edit the draft. ===


## Step 4: Resume the Graph with Your Decision

The graph is **still paused**. Now you send your human decision back.
- `Command(resume='approve')` tells the graph to keep the AI draft.
- `Command(resume='My own answer here')` replaces the draft with your text.

**Try both!** First run with `'approve'`, then restart from Step 3 and try editing.

**If Cell 5 does not prompt for input:** restart the notebook kernel once, then run Cells 1 to 5 again. This clears any old test state from memory.

In [6]:
# Cell 5: Resume the paused graph with your decision

# Type 'approve' to keep the AI draft, or type your own answer to replace it.
my_decision = builtins.input(
    "Type 'approve' to accept the draft, or enter a replacement answer: "
).strip()

if not my_decision:
    print("No input detected. The graph is still paused.")
    print("Run this cell again and type 'approve' or your own edited answer.")
else:
    final_result = app.invoke(Command(resume=my_decision), config=config)

    print('\n=== GRAPH RESUMED AND FINISHED ===')
    if my_decision.lower() == 'approve':
        print("You approved the AI draft.")
    else:
        print("You replaced the AI draft with your own answer.")
    print(f"Your decision was: '{my_decision}'")
    print(f"Final answer: {final_result['final_answer']}")

[human_review] Human responded with: reject

=== GRAPH RESUMED AND FINISHED ===
You replaced the AI draft with your own answer.
Your decision was: 'reject'
Final answer: 
reject



## Exercises (Try These!)

Now that you understand the flow, try these changes to deepen your understanding:

### Exercise 1: Edit the draft instead of approving
Go back to Cell 5 and change `my_decision` from `'approve'` to your own answer, e.g.:
```python
my_decision = 'LangGraph is a framework for building multi-step AI agents.'
```
Then re-run Cells 3, 4, 5 in order. Notice how `final_answer` now uses YOUR text.

### Exercise 2: Change the question
In Cell 4, change the question from `'What is LangGraph?'` to something else, e.g.:
```python
result = app.invoke({'question': 'Why is HITL important?'}, config=config)
```
Re-run Cells 3, 4, 5. The draft changes because the question changed.

### Exercise 3: Use a different thread_id
Change the `thread_id` in Cell 4:
```python
config = {'configurable': {'thread_id': 'experiment-2'}}
```
This creates a separate HITL session. You can have multiple paused sessions at once.

### Exercise 4: Remove the checkpointer and see what breaks
In Cell 3, try:
```python
app = graph.compile()  # no checkpointer!
```
Re-run Cells 3 and 4. You will get an error because `interrupt()` needs a checkpointer to save the pause point.

### Exercise 5: Add a third node
Add a `format_answer` node between `human_review` and `END` that wraps the final answer in a box:
```python
def format_answer(state: State):
    return {'final_answer': f'\n{state["final_answer"]}\n'}
```
Wire it in: `human_review` → `format_answer` → `END`.